In [20]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from typing import List, Tuple, Dict, Optional
from openpyxl import load_workbook
import os

# OR-Tools for Constraint Programming
from ortools.sat.python import cp_model

In [21]:
Rooms = pd.read_excel("../data/Raw Data/Data.xlsx" , sheet_name="Rooms")
Courses = pd.read_excel("../data/Raw Data/Data.xlsx" , sheet_name="Courses" )
Doctors = pd.read_excel("../data/Raw Data/Data.xlsx" , sheet_name="Doctors" )
Divisions = pd.read_excel("../data/Raw Data/Data.xlsx" , sheet_name="Division" )

In [22]:
Rooms.head(10)

,Room_ID,Room,Capacity,Type
0,R1,Room 1,30,Lab
1,R2,Room 2,30,Lab
2,R3,Room 3,30,Lab
3,R4,Room 4,30,Lab
4,R5,Room 5,30,Lab
5,T8,Terrace 8,250,Lecture
6,T4,terrace4,100,Lecture
7,H16,Hall 16,100,Lecture


In [23]:
Courses.head(10)

,Course_ID,Course_Name,Department,Major,Days,Hours_per_day,Instructor_ID,Year,Type,Duration
0,C01,Mathematics (0),IT,IT,2,1,I12,1,Lecture,1 Hour
1,C02,Mathematics (1),IT,IT,2,1,I12,1,Lecture,1 Hour
2,C03,Electronics,IT,IT,2,1,I13,1,Lecture,1 Hour
3,C04,Discrete Mathematics,IT,IT,2,1,I16,1,Lecture,1 Hour
4,C05,Introduction to Computers,IT,IT,2,1,I18,1,Lecture,1 Hour
5,C06,Mathematics-3,IT,IT,2,1,I16,2,Lecture,1 Hour
6,C07,Computer Networks Technology,IT,IT,2,1,I14,2,Lecture,1 Hour
7,C08,Introduction to Software Engineering,IT,IT,2,1,I15,2,Lecture,1 Hour
8,C09,Object Oriented Programming,IT,IT,2,1,I03,2,Lecture,1 Hour
9,C10,Probability and Statistics-2,IT,IT,2,1,I05,2,Lecture,1 Hour


In [24]:
Doctors.head(10)

,Instructor_ID,Instructor_Name,Department,Day,Start_Time,End_Time,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9
0,I01,Dr. Shimaa Mosaad,IT,Saturday,08:00:00,09:00:00,NaN,NaN,NaN,NaN
1,I01,Dr. Shimaa Mosaad,IT,Tuesday,08:00:00,09:00:00,NaN,NaN,NaN,NaN
2,I02,Dr. Kamal Hamza,IT,Sunday,14:00:00,15:00:00,NaN,NaN,NaN,NaN
3,I02,Dr. Kamal Hamza,IT,Wednesday,14:00:00,15:00:00,NaN,NaN,NaN,NaN
4,I03,Dr. Yasser,IT,Monday,14:00:00,15:00:00,NaN,NaN,NaN,NaN
5,I03,Dr. Yasser,IT,Thursday,14:00:00,15:00:00,NaN,NaN,NaN,NaN
6,I04,Dr. Adel Fathy Khalifa,IT,Saturday,12:00:00,13:00:00,NaN,NaN,NaN,NaN
7,I04,Dr. Adel Fathy Khalifa,IT,Tuesday,12:00:00,13:00:00,NaN,NaN,NaN,NaN
8,I05,Dr. Abdelbaset,IT,Monday,15:00:00,16:00:00,NaN,NaN,NaN,NaN
9,I05,Dr. Abdelbaset,IT,Thursday,15:00:00,16:00:00,NaN,NaN,NaN,NaN


In [25]:
Divisions.head(20)

,Num_ID,Department,Major,Year,StudentNum
0,D01,IT,IT,1,219
1,D02,IT,IT,2,235
2,D03,IT,IT,3,264
3,D04,IT,IT,4,299
4,D05,BA,BA,1,125
5,D06,BA,BA,2,127
6,D07,BA,BA,3,65
7,D08,BA,AC,3,117
8,D09,BA,BA,4,54
9,D10,BA,AC,4,77


In [26]:
def parse_time(t):
    h, m = map(int, str(t).split(":"))
    return h * 60 + m

def time_ranges_overlap(start1, end1, start2, end2):
    return not (end1 <= start2 or end2 <= start1)

def parse_time_extended(time_str):
    try:
        if isinstance(time_str, str):
            time_str = time_str.strip().upper()
            
            is_pm = 'PM' in time_str
            is_am = 'AM' in time_str
            
            time_str = time_str.replace('AM', '').replace('PM', '').strip()
            
            parts = time_str.split(':')
            hours = int(parts[0])
            minutes = int(parts[1]) if len(parts) > 1 else 0
            
            if hours == 12:
                if is_am:
                    hours = 0  
            elif is_pm and hours != 12:
                hours += 12  
            
            return hours * 60 + minutes
        return 0
    except:
        return 0

def minutes_to_time_str(minutes):
    if minutes == 0:
        return "12:00 AM"
    hours = minutes // 60
    mins = minutes % 60
    hours = hours % 24
   
    if hours == 0:
        hour_12 = 12
        period = "AM"
    elif hours < 12:
        hour_12 = hours
        period = "AM"
    elif hours == 12:
        hour_12 = 12
        period = "PM"
    else:
        hour_12 = hours - 12
        period = "PM"
    
    return f"{hour_12}:{mins:02d} {period}"

def is_time_in_range(time_minutes, start_minutes, end_minutes):
    if end_minutes < start_minutes: 
        return time_minutes >= start_minutes or time_minutes <= end_minutes
    return start_minutes <= time_minutes <= end_minutes

def time_to_minutes(time_value):
    from datetime import time as dt_time
    if isinstance(time_value, (int, float)):
        return int(time_value)

    if isinstance(time_value, dt_time):
        return time_value.hour * 60 + time_value.minute
    
    if isinstance(time_value, str):
        result = parse_time_extended(time_value)
        if result > 0:
            return result
        
        try:
            return parse_time(time_value)
        except:
            return 0
    
    return 0

print("Helper functions defined")

Helper functions defined


In [27]:
# Prepare data structures
room_dict = {row['Room_ID']: {'capacity': row['Capacity'], 'type': row['Type']} 
             for _, row in Rooms.iterrows()}

course_dict = {row['Course_ID']: {
    'instructor': row['Instructor_ID'],
    'days': row['Days'],
    'hours_per_day': row['Hours_per_day'],
    'type': row['Type'],
    'year': row['Year'],
    'major': row['Major'],
    'department': row['Department']
} for _, row in Courses.iterrows()}

doctor_availability = {}
for _, row in Doctors.iterrows():
    inst_id = row['Instructor_ID']
    day = row['Day']
    start = row['Start_Time']
    end = row['End_Time']
    start_minutes = time_to_minutes(start)
    end_minutes = time_to_minutes(end)
    
    if inst_id not in doctor_availability:
        doctor_availability[inst_id] = {}
    if day not in doctor_availability[inst_id]:
        doctor_availability[inst_id][day] = []
    doctor_availability[inst_id][day].append((start_minutes, end_minutes))

division_dict = {row['Num_ID']: {
    'students': row['StudentNum'],
    'year': row['Year'],
    'major': row['Major'],
    'department': row['Department']
} for _, row in Divisions.iterrows()}

lecture_rooms = [r for r in room_dict.keys() if room_dict[r]['type'] == 'Lecture']
lab_rooms = [r for r in room_dict.keys() if room_dict[r]['type'] == 'Lab']

print(f"Data prepared: {len(course_dict)} courses, {len(room_dict)} rooms, {len(division_dict)} divisions")
print(f"Lecture rooms: {len(lecture_rooms)}, Lab rooms: {len(lab_rooms)}")

Data prepared: 53 courses, 8 rooms, 10 divisions
Lecture rooms: 3, Lab rooms: 5


In [28]:
class SchedulingCP:
    """Constraint Programming scheduler using OR-Tools CP-SAT."""
    
    def __init__(self, courses_df, rooms_df, doctors_df, divisions_df, 
                 time_limit_seconds=300, use_optimization=True):
        """Initialize the CP scheduler.
        
        Args:
            courses_df: DataFrame with course information
            rooms_df: DataFrame with room information
            doctors_df: DataFrame with doctor availability
            divisions_df: DataFrame with division information
            time_limit_seconds: Maximum time to search for solution
            use_optimization: Whether to optimize for better solutions
        """
        self.courses_df = courses_df
        self.rooms_df = rooms_df
        self.doctors_df = doctors_df
        self.divisions_df = divisions_df
        self.time_limit_seconds = time_limit_seconds
        self.use_optimization = use_optimization
        
        self.prepare_data()
    
    def prepare_data(self):
        """Prepare data structures for CP solver."""
        # Get all courses
        self.courses = self.courses_df['Course_ID'].tolist()
        
        # Build course-division pairs (ONE division per course)
        self.divisions = []
        for _, course_row in self.courses_df.iterrows():
            course_id = course_row['Course_ID']
            year = course_row['Year']
            major = course_row['Major']
            dept = course_row['Department']
            
            matching_divs = self.divisions_df[
                (self.divisions_df['Year'] == year) &
                (self.divisions_df['Major'] == major) &
                (self.divisions_df['Department'] == dept)
            ]
            
            if not matching_divs.empty:
                # Pick the first matching division (largest student group)
                best_div = matching_divs.sort_values('StudentNum', ascending=False).iloc[0]
                self.divisions.append((course_id, best_div['Num_ID']))
            else:
                # No matching division found - try matching by Year and Department only
                fallback_divs = self.divisions_df[
                    (self.divisions_df['Year'] == year) &
                    (self.divisions_df['Department'] == dept)
                ]
                if not fallback_divs.empty:
                    best_div = fallback_divs.sort_values('StudentNum', ascending=False).iloc[0]
                    self.divisions.append((course_id, best_div['Num_ID']))
                else:
                    print(f"Warning: No matching division for course {course_id} (Year={year}, Major={major}, Dept={dept})")
        
        # Get available days
        if len(self.doctors_df) > 0 and 'Day' in self.doctors_df.columns:
            self.days = self.doctors_df['Day'].unique().tolist()
        else:
            self.days = ['Sunday', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Saturday']
        
        self.all_rooms = self.rooms_df['Room_ID'].tolist()
        self.lecture_rooms = self.rooms_df[self.rooms_df['Type'] == 'Lecture']['Room_ID'].tolist()
        self.lab_rooms = self.rooms_df[self.rooms_df['Type'] == 'Lab']['Room_ID'].tolist()
        
        # Pre-compute room capacities (room_id -> capacity) for strict capacity constraints
        self.room_capacity = {}
        for _, row in self.rooms_df.iterrows():
            self.room_capacity[row['Room_ID']] = int(row['Capacity'])
        
        # Define time slots (in minutes since midnight)
        self.time_slots = list(range(8 * 60, 17 * 60 + 1, 60))  # 8 AM to 5 PM, hourly
        
        print(f"Course-division pairs: {len(self.divisions)}")
        print(f"Days: {self.days}")
        print(f"Time slots: {len(self.time_slots)} slots")
        print(f"Room capacities: {self.room_capacity}")
    
    def build_model(self):
        """Build the CP-SAT model with all constraints."""
        model = cp_model.CpModel()
        
        # Create decision variables
        # For each course-division pair, we need 'days' number of assignments
        # Each assignment needs: day, room, start_time
        
        # Index mappings
        self.day_indices = {day: i for i, day in enumerate(self.days)}
        self.room_indices = {room: i for i, room in enumerate(self.all_rooms)}
        self.time_indices = {time: i for i, time in enumerate(self.time_slots)}
        
        # Create variables for each assignment
        # assignment_vars[pair_idx][day_idx] = (room_idx, time_idx)
        self.assignment_vars = {}
        
        # For each course-division pair
        for pair_idx, (course_id, div_id) in enumerate(self.divisions):
            course_info = self.courses_df[self.courses_df['Course_ID'] == course_id].iloc[0]
            days_needed = int(course_info['Days'])
            
            # Create variables for each required day
            self.assignment_vars[pair_idx] = {}
            
            for day_idx in range(days_needed):
                # Day variable
                day_var = model.NewIntVar(0, len(self.days) - 1, f"day_{pair_idx}_{day_idx}")
                
                # Room variable
                room_var = model.NewIntVar(0, len(self.all_rooms) - 1, f"room_{pair_idx}_{day_idx}")
                
                # Time variable
                time_var = model.NewIntVar(0, len(self.time_slots) - 1, f"time_{pair_idx}_{day_idx}")
                
                self.assignment_vars[pair_idx][day_idx] = {
                    'day': day_var,
                    'room': room_var,
                    'time': time_var,
                    'course_id': course_id,
                    'div_id': div_id
                }
        
        # Add constraints
        self._add_constraints(model)
        
        return model
    
    def _add_constraints(self, model):
        """Add all constraints to the model."""
        
        # For each assignment, get course and division info
        for pair_idx, assignments in self.assignment_vars.items():
            course_id, div_id = self.divisions[pair_idx]
            course_info = self.courses_df[self.courses_df['Course_ID'] == course_id].iloc[0]
            div_info = self.divisions_df[self.divisions_df['Num_ID'] == div_id].iloc[0]
            
            instructor_id = course_info['Instructor_ID']
            course_type = course_info['Type']
            hours_per_day = int(course_info['Hours_per_day'])
            duration_minutes = hours_per_day * 60
            total_students = int(div_info['StudentNum'])
            # Hybrid Rotation Logic: Only 50% of students need a room at once
            students_requiring_room = (total_students + 1) // 2
            
            # Room capacity constraint: only allow rooms that can fit the students.
            # Filter by course type (Lecture/Lab) AND by capacity >= students_requiring_room.
            if course_type == 'Lecture':
                candidate_rooms = self.lecture_rooms
            else:
                candidate_rooms = self.lab_rooms
            
            suitable_rooms = [self.room_indices[r] for r in candidate_rooms 
                            if self.room_capacity.get(r, 0) >= students_requiring_room]
            
            # If no single room fits, allow only the largest room(s) of that type (last resort)
            if not suitable_rooms:
                sorted_by_capacity = sorted(candidate_rooms, 
                    key=lambda r: self.room_capacity.get(r, 0), reverse=True)
                if sorted_by_capacity:
                    largest_cap = self.room_capacity.get(sorted_by_capacity[0], 0)
                    suitable_rooms = [self.room_indices[r] for r in sorted_by_capacity 
                                    if self.room_capacity.get(r, 0) == largest_cap]
                    print(f"Warning: No room fits {students_requiring_room} students for {course_id}/{div_id}. "
                          f"Using largest capacity {largest_cap} as fallback.")
            
            # Get available days for instructor
            available_day_indices = []
            if instructor_id in doctor_availability:
                for day in doctor_availability[instructor_id].keys():
                    if day in self.day_indices:
                        available_day_indices.append(self.day_indices[day])
            
            if not available_day_indices or len(available_day_indices) < days_needed:
                available_day_indices = list(range(len(self.days)))
            
            # Constraint: Each assignment must use a room with sufficient capacity (and correct type)
            for day_idx, assignment in assignments.items():
                # Room must be in suitable_rooms (type + capacity >= students_requiring_room)
                model.AddAllowedAssignments([assignment['room']], [(r,) for r in suitable_rooms])
                
                # Day must be available for instructor
                model.AddAllowedAssignments([assignment['day']], [(d,) for d in available_day_indices])
                
                # Time must be valid (not too late for duration)
                # Ensure start_time + duration <= end of day (17:00 = 1020 minutes)
                max_time_idx = len(self.time_slots) - 1
                for time_idx, start_time in enumerate(self.time_slots):
                    end_time = start_time + duration_minutes
                    if end_time > 17 * 60:  # After 5 PM
                        model.Add(assignment['time'] != time_idx)
        
        # Constraint: No conflicts
        # For each pair of assignments, if they conflict, add constraint
        for pair_idx1, assignments1 in self.assignment_vars.items():
            course_id1, div_id1 = self.divisions[pair_idx1]
            course_info1 = self.courses_df[self.courses_df['Course_ID'] == course_id1].iloc[0]
            instructor_id1 = course_info1['Instructor_ID']
            hours_per_day1 = int(course_info1['Hours_per_day'])
            duration1 = hours_per_day1 * 60
            
            for day_idx1, assignment1 in assignments1.items():
                for pair_idx2, assignments2 in self.assignment_vars.items():
                    if pair_idx1 >= pair_idx2:
                        continue
                    
                    course_id2, div_id2 = self.divisions[pair_idx2]
                    course_info2 = self.courses_df[self.courses_df['Course_ID'] == course_id2].iloc[0]
                    instructor_id2 = course_info2['Instructor_ID']
                    hours_per_day2 = int(course_info2['Hours_per_day'])
                    duration2 = hours_per_day2 * 60
                    
                    for day_idx2, assignment2 in assignments2.items():
                        # Room conflict: same room, same day, overlapping times
                        room_conflict = model.NewBoolVar(f"room_conflict_{pair_idx1}_{day_idx1}_{pair_idx2}_{day_idx2}")
                        model.Add(assignment1['room'] == assignment2['room']).OnlyEnforceIf(room_conflict)
                        model.Add(assignment1['room'] != assignment2['room']).OnlyEnforceIf(room_conflict.Not())
                        
                        same_day = model.NewBoolVar(f"same_day_{pair_idx1}_{day_idx1}_{pair_idx2}_{day_idx2}")
                        model.Add(assignment1['day'] == assignment2['day']).OnlyEnforceIf(same_day)
                        model.Add(assignment1['day'] != assignment2['day']).OnlyEnforceIf(same_day.Not())
                        
                        # Time overlap constraint
                        # We need to check if time ranges overlap
                        # This is complex, so we'll use a simpler approach:
                        # If same room and same day, times must be different
                        # For more accuracy, we'd need to check actual time ranges
                        
                        # If same room AND same day, times must be different (no double-booking)
                        both_room_day = model.NewBoolVar(f"both_rd_{pair_idx1}_{day_idx1}_{pair_idx2}_{day_idx2}")
                        model.AddBoolAnd([room_conflict, same_day]).OnlyEnforceIf(both_room_day)
                        model.AddBoolOr([room_conflict.Not(), same_day.Not()]).OnlyEnforceIf(both_room_day.Not())
                        model.Add(assignment1['time'] != assignment2['time']).OnlyEnforceIf(both_room_day)
                        
                        # Instructor conflict: same instructor + same day => different times
                        if instructor_id1 == instructor_id2:
                            model.Add(assignment1['time'] != assignment2['time']).OnlyEnforceIf(same_day)
                        
                        # Division conflict: same division + same day => different times
                        if div_id1 == div_id2:
                            model.Add(assignment1['time'] != assignment2['time']).OnlyEnforceIf(same_day)
        
        # Constraint: All days for a course must be different
        for pair_idx, assignments in self.assignment_vars.items():
            if len(assignments) > 1:
                day_vars = [a['day'] for a in assignments.values()]
                # All days must be different
                model.AddAllDifferent(day_vars)
    
    def solve(self):
        """Solve the constraint programming problem."""
        print("Building CP model...")
        model = self.build_model()
        
        print("Creating solver...")
        solver = cp_model.CpSolver()
        solver.parameters.max_time_in_seconds = self.time_limit_seconds
        
        # Add callback for solution tracking
        solution_collector = SolutionCollector(self.assignment_vars, self.divisions, 
                                             self.days, self.all_rooms, self.time_slots,
                                             self.courses_df, self.divisions_df)
        
        print(f"Solving (time limit: {self.time_limit_seconds}s)...")
        status = solver.Solve(model, solution_collector)
        
        status_name = {cp_model.OPTIMAL: 'OPTIMAL', cp_model.FEASIBLE: 'FEASIBLE',
                     cp_model.INFEASIBLE: 'INFEASIBLE', cp_model.UNKNOWN: 'UNKNOWN (time limit?)'}
        print(f"Solver status: {status_name.get(status, status)}")
        if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
            print(f"Solution found!")
            return solution_collector.get_best_solution()
        if status == cp_model.INFEASIBLE:
            print("Model is infeasible: constraints cannot all be satisfied.")
        else:
            print("No solution yet. Try increasing time_limit_seconds (e.g. 600).")
        return None


class SolutionCollector(cp_model.CpSolverSolutionCallback):
    """Collect solutions during search."""
    
    def __init__(self, assignment_vars, divisions, days, rooms, time_slots, 
                 courses_df, divisions_df):
        cp_model.CpSolverSolutionCallback.__init__(self)
        self.assignment_vars = assignment_vars
        self.divisions = divisions
        self.days = days
        self.rooms = rooms
        self.time_slots = time_slots
        self.courses_df = courses_df
        self.divisions_df = divisions_df
        self.solutions = []
    
    def on_solution_callback(self):
        """Called when a solution is found."""
        solution = []
        
        for pair_idx, assignments in self.assignment_vars.items():
            course_id, div_id = self.divisions[pair_idx]
            
            for day_idx, assignment in assignments.items():
                day = self.days[self.Value(assignment['day'])]
                room = self.rooms[self.Value(assignment['room'])]
                start_time = self.time_slots[self.Value(assignment['time'])]
                
                course_info = self.courses_df[self.courses_df['Course_ID'] == course_id].iloc[0]
                hours_per_day = int(course_info['Hours_per_day'])
                end_time = start_time + (hours_per_day * 60)
                
                solution.append({
                    'Day': day,
                    'Course_ID': course_id,
                    'Instructor_ID': course_info['Instructor_ID'],
                    'Group_ID': div_id,
                    'Room_ID': room,
                    'Time_Slot': f"{day}_{start_time}_{end_time}",
                    'Start_Time': start_time,
                    'End_Time': end_time,
                    'Duration': hours_per_day * 60
                })
        
        self.solutions.append(solution)
    
    def get_best_solution(self):
        """Return the best solution found."""
        if self.solutions:
            return self.solutions[-1]  # Return last (best) solution
        return None


print("Constraint Programming scheduler class defined")

Constraint Programming scheduler class defined


In [29]:
# Create and run CP scheduler
cp_scheduler = SchedulingCP(
    courses_df=Courses,
    rooms_df=Rooms,
    doctors_df=Doctors,
    divisions_df=Divisions,
    time_limit_seconds=300,  # 5 minutes
    use_optimization=True
)

best_schedule = cp_scheduler.solve()

Course-division pairs: 53
Days: ['Saturday', 'Tuesday', 'Sunday', 'Wednesday', 'Monday', 'Thursday']
Time slots: 10 slots
Room capacities: {'R1': 30, 'R2': 30, 'R3': 30, 'R4': 30, 'R5': 30, 'T8': 250, 'T4': 100, 'H16': 100}
Building CP model...
Creating solver...
Solving (time limit: 300s)...
No solution found. Status: 3


In [30]:
if best_schedule:
    schedule_data = []
    for assignment in best_schedule:
        course_id = assignment['Course_ID']
        instructor_id = assignment['Instructor_ID']
        group_id = assignment['Group_ID']
        room_id = assignment['Room_ID']
        day = assignment['Day']
        
        course_info = Courses[Courses['Course_ID'] == course_id].iloc[0]
        instructor_info = Doctors[Doctors['Instructor_ID'] == instructor_id].iloc[0]
        division_info = Divisions[Divisions['Num_ID'] == group_id].iloc[0]
        room_info = Rooms[Rooms['Room_ID'] == room_id].iloc[0]
        
        start_time_min = assignment.get('Start_Time', 0)
        end_time_min = assignment.get('End_Time', 0)
        
        start_time_str = minutes_to_time_str(start_time_min)
        end_time_str = minutes_to_time_str(end_time_min)
        
        schedule_data.append({
            'Day': day,
            'Course_Name': course_info['Course_Name'],
            'Instructor_Name': instructor_info['Instructor_Name'],
            'Students': division_info['StudentNum'],
            'Room': room_info['Room'],
            'Start_Time': start_time_str,
            'End_Time': end_time_str,
            'Department': course_info['Department'],
            'Major': course_info['Major']
        })

    Schedule = pd.DataFrame(schedule_data)
    
    print(f"\nGenerated schedule with {len(Schedule)} assignments:")
    print(Schedule.head(20))
    print(f"\nTotal assignments: {len(Schedule)}")
    
    # Verify room capacity for each assignment
    print("\n--- Room capacity verification ---")
    over_count = 0
    for assignment in best_schedule:
        room_id = assignment['Room_ID']
        div_id = assignment['Group_ID']
        cap = room_dict.get(room_id, {}).get('capacity', 0)
        div_info = Divisions[Divisions['Num_ID'] == div_id].iloc[0]
        students = (int(div_info['StudentNum']) + 1) // 2  # hybrid rotation
        if cap < students:
            over_count += 1
            print(f"  OVER: {assignment['Course_ID']} -> {room_id} (capacity {cap}, students {students})")
    if over_count == 0:
        print("  All assignments respect room capacity.")
    else:
        print(f"  Warning: {over_count} assignment(s) exceed room capacity.")
else:
    print("No solution found. The problem may be infeasible or time limit was too short.")

No solution found. The problem may be infeasible or time limit was too short.


In [31]:
if best_schedule:
    processed_data_dir = "../../data/Processed Data"
    os.makedirs(processed_data_dir, exist_ok=True)

    output_path = "../../data/Processed Data/Schedule_Output_CP.xlsx"
    Schedule.to_excel(output_path, sheet_name='Schedule', index=False)
    print(f"Schedule saved to {output_path} successfully!")